In [ ]:
import os
from pathlib import Path

import pandas as pd
import yaml

from pynxtools_apm.examples.deu_berlin_kuehbach.utils import get_project_id

print(os.getcwd())
with open(f"{os.getcwd()}{os.sep}/config.yaml", encoding="utf-8") as fp:
    config: dict[str, str] = yaml.safe_load(fp)
for task, location in config.items():
    print(f"{task}, {location}")

# os.makedirs(f"{trg_directory.replace('/decompressed', '/pynxtools')}", exist_ok=True)
# os.listdir(f"{trg_directory.replace('/decompressed', '')}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{config['data']}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (1, 160)

with open(f"{config['data']}{os.sep}aaa_apm_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

## Analyze the run-through log files for warnings and errors to support parser coverage and bug fixing

In [ ]:
count: int = 0
issues: list[tuple[str, str]] = []
# pattern = r'(?:[^;]*;){3}ERROR'
issue_type = "ERROR"
# issue_type = "WARNING"

for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.id)
        if project_range[0] <= int(project_id) <= project_range[1]:
            # if row.legal == "1" and row.use == "1":
            # if row.legal == "0" or row.use == "0":
            if True:
                if project_id in nomad_project_names:
                    if project_id in ("010"):
                        print(
                            f"{row.project_name} uses an ancient nowadays unsupported compression algorithm in an RHIT file, WONTFIX"
                        )
                        continue
                    elif project_id in ("013"):
                        print(
                            f"{row.project_name} throws an uproot deserialization issue, WONTFIX"
                        )
                        continue
                    elif project_id in ("017", "159"):
                        print(
                            f"{row.project_name} has an ambiguous RRNG ion section, INSPECT"
                        )
                        continue
                    elif project_id in ("113"):
                        print(
                            f"{row.project_name} has CSV files representing mass spectra, not POS file content, WONTFIX"
                        )
                        continue
                    elif project_id in ("021", "026", "036", "076", "114"):
                        print(f"{row.project_name} has unsupported CSV format, INSPECT")
                        continue
                    elif project_id in ("046", "058"):
                        print(
                            f"{row.project_name} has incompatible strides in an APT file, WONTFIX"
                        )
                        continue
                    elif project_id in ("075", "151"):
                        print(
                            f"{row.project_name} has corrupted RHIT file, chunk length polling issues, WONTFIX"
                        )
                        continue
                    elif project_id in ("144"):
                        print(
                            f"{row.project_name} index out of bounds for an RHIT reader, WONTFIX"
                        )
                        continue

                    log_file_path = f"{config['pynxtools']}{os.sep}run01{os.sep}{project_id}.{row.project_name}.paper.run01.csv"
                    if os.path.exists(log_file_path):
                        # print(log_file_path)
                        lines: list[str] = (
                            Path(log_file_path).read_text(encoding="utf-8").splitlines()
                        )
                        for line in lines:
                            tokenized = line.split(";")
                            if tokenized[2] == issue_type:
                                issues.append((row.project_name, tokenized[3:]))
                        del lines

# 017, fixed, the rrng file is malformed, see Range71 is a "Mo:1C:1" while it should be "Mo:1 C:1"
# 159, fixed
# 026, fixed
# 036, fixed
# postpone simulations
# 021, bel_imec_lueken needs own parser imec_adapts ad hoc csv-based format with additional metadata parsing required
# 076, fin_aalto_shaikh needs own parser, rouen_rrm ad hoc csv-based format with additional metadata parsing required
# 113, usa_boulder_chiaramonti, needs own parser ad hoc csv with additional metadata parsing required
# 114, usa_buffalo_mukherjee needs own parser, oscar branson's ad hoc csv-based epos inspired format with additional metadata parsing required
# 140, usa_idaho_falls_tang, HiPerClust needs own parser

In [ ]:
unique_issues: set[str] = set()
for project_name, issue in issues:
    # if project_name == "usa_boulder_chiaramonti":  # "gbr_cambridge_posap":
    # if not (issue[0].startswith("No content to parse") or issue[0].endswith("for an original research article")):
    #     unique_issues.add(f"{issue}")
    # if "section" in issue:
    print(f"{project_name} >>>> {issue}")
# for issue in unique_issues:
#     print(issue)

# errors within the publicly shared datasets
# unable to load project, details via OpenAlex, unable to load json, we can ignore these as we have here data from personal communication or otherwise the bib covering for the missing pieces

# warnings within the publicly shared datasets
# ignore, "No content to parse", expected cases of rows that list only those files we ignore, so fine
# ignore, "for an original research article", expected, we no longer wish to report always dataset and paper because not each dataset has clearly a paper mentioned
# TODO, gbr_cambridge_posap.1.3 that is issue which requires inspection malformed pos also causing pos_file.get_reconstructed and get_mass_to_charge
# TODO, skipping a non-deflate-compressed CRunHeader
# fixed, "apt_file.get_named_quantity or "Unable to get_named_quantity", clarified by edits in ifes parser

# errors in addition in non-publicly shared datasets
# fixed, deu_bochum_muench 157, renamed to deu_karlsruhe_muench_ghost remains 157, adding references in the bibtex and reprocessing fixed it

# warnings in addition in non-publicly shared datasets
# missing default plot in usa_boulder_chiaramonti caused by CSV not representing ion positions but mass spectra

***